In [1]:
# ==========================================
# Google Colab Setup & Installation Block
# ==========================================
import subprocess
import sys

def install_and_import(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Install required libraries
print("--- Installing required libraries ---")
install_and_import("spacy")
install_and_import("gtts")

# Download spaCy English language model
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

# ==========================================
# Main Script
# ==========================================
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation
from heapq import nlargest
from gtts import gTTS
from IPython.display import Audio, display

# 1. Initialize spaCy NLP Model
print("\n--- Initializing NLP Model ---")
nlp = spacy.load('en_core_web_sm')

# 2. Extractive Summarization Function
def summarize_text(text, summary_ratio=0.3):
    doc = nlp(text)

    stopwords = list(STOP_WORDS)
    punct = punctuation + '\n'

    word_frequencies = {}
    for word in doc:
        word_text = word.text.lower()
        if word_text not in stopwords and word_text not in punct:
            if word_text not in word_frequencies:
                word_frequencies[word_text] = 1
            else:
                word_frequencies[word_text] += 1

    if not word_frequencies:
        return "Text is too short or contains only stop words."

    max_frequency = max(word_frequencies.values())
    for word in word_frequencies:
        word_frequencies[word] = word_frequencies[word] / max_frequency

    sentence_tokens = list(doc.sents)
    sentence_scores = {}

    for sent in sentence_tokens:
        for word in sent:
            word_text = word.text.lower()
            if word_text in word_frequencies:
                if sent not in sentence_scores:
                    sentence_scores[sent] = word_frequencies[word_text]
                else:
                    sentence_scores[sent] += word_frequencies[word_text]

    select_length = int(len(sentence_tokens) * summary_ratio)
    select_length = max(1, select_length)

    summary_sentences = nlargest(select_length, sentence_scores, key=sentence_scores.get)
    summary_sentences = sorted(summary_sentences, key=lambda x: x.start)

    final_summary = " ".join([sent.text for sent in summary_sentences])
    return final_summary

# 3. Text-to-Speech Function for Colab
def text_to_speech_colab(text, filename="summary_audio.mp3"):
    print("\n--- Converting Summary to Speech ---")
    tts = gTTS(text=text, lang='en', slow=False)
    tts.save(filename)
    print(f"Audio saved successfully as '{filename}'!")
    return filename

# 4. Execution Pipeline
if __name__ == "__main__":
    # Sample input text
    sample_text = """
    Natural Language Processing (NLP) is a subfield of artificial intelligence, computer science,
    and linguistics focused on the interactions between computers and human language.
    It enables computers to understand, interpret, and manipulate human speech and text.
    NLP draws from many disciplines, including computer science and computational linguistics,
    aiming to fill the gap between human communication and machine understanding.
    Automatic text summarization is a crucial application of NLP. It reduces the length of a text
    while preserving its core meaning and key information. There are primarily two approaches to text
    summarization: extractive and abstractive. Extractive summarization selects the most critical
    sentences directly from the source document, whereas abstractive summarization generates entirely
    new phrases to convey the core meaning. Combining NLP text summarization with Text-to-Speech (TTS)
    technology allows systems to read out condensed news, articles, or research abstracts efficiently,
    greatly enhancing accessibility and saving time for users on the go.
    """

    print("\n" + "="*40)
    print("ORIGINAL TEXT:")
    print("="*40)
    print(sample_text.strip())

    # Generate Summary
    summary = summarize_text(sample_text, summary_ratio=0.3)

    print("\n" + "="*40)
    print("GENERATED SUMMARY:")
    print("="*40)
    print(summary)

    # Convert summary to speech audio and play directly in Colab
    audio_file = text_to_speech_colab(summary)

    print("\n--- Play Audio Output Below ---")
    display(Audio(audio_file, autoplay=True))

--- Installing required libraries ---

--- Initializing NLP Model ---

ORIGINAL TEXT:
Natural Language Processing (NLP) is a subfield of artificial intelligence, computer science, 
    and linguistics focused on the interactions between computers and human language. 
    It enables computers to understand, interpret, and manipulate human speech and text. 
    NLP draws from many disciplines, including computer science and computational linguistics, 
    aiming to fill the gap between human communication and machine understanding. 
    Automatic text summarization is a crucial application of NLP. It reduces the length of a text 
    while preserving its core meaning and key information. There are primarily two approaches to text 
    summarization: extractive and abstractive. Extractive summarization selects the most critical 
    sentences directly from the source document, whereas abstractive summarization generates entirely 
    new phrases to convey the core meaning. Combining NLP t